# 💨 Chương 4 — Dataset 1: Beijing Air Quality — Multivariate LSTM
## Kỹ thuật: Multivariate Time Series + LSTM + Sliding Window

**Pipeline:** EDA → Correlation Analysis → Sliding Window → Multivariate LSTM → Train → Forecast Visualization

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import glob, os, warnings

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11,
                     'axes.titlesize': 13, 'axes.titleweight': 'bold'})

DATA_DIR = '../data/beijing_air_quality'
SAVE_DIR = '../results/beijing_air_quality'
os.makedirs(SAVE_DIR, exist_ok=True)

LOOK_BACK  = 24   # 24 giờ ngữ cảnh
EPOCHS     = 30
BATCH_SIZE = 64
TARGET_COL = 'pm2.5'

print(f'TF: {tf.__version__}')

## 📂 1. Load & Tổng quan Data

In [ ]:
csv_files = glob.glob(f'{DATA_DIR}/**/*.csv', recursive=True) + glob.glob(f'{DATA_DIR}/*.csv')
print('Found:', csv_files)

# Load and combine if multiple files
dfs = [pd.read_csv(f) for f in csv_files]
df = pd.concat(dfs, ignore_index=True) if len(dfs) > 1 else dfs[0]

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Detect target column (PM2.5)
pm_cols = [c for c in df.columns if 'pm' in c.lower() or 'pm2' in c.lower()]
if pm_cols:
    TARGET_COL = pm_cols[0]
    print(f'Target column detected: {TARGET_COL}')

# Drop NaN in target
df = df.dropna(subset=[TARGET_COL])

# Build datetime index if possible
date_cols = [c for c in df.columns if any(k in c.lower() for k in ['year','month','day','date','hour'])]
if 'year' in df.columns and 'month' in df.columns and 'day' in df.columns:
    try:
        if 'hour' in df.columns:
            df['datetime'] = pd.to_datetime(df[['year','month','day','hour']])
        else:
            df['datetime'] = pd.to_datetime(df[['year','month','day']])
        df = df.set_index('datetime')
        print('Datetime index set successfully')
    except:
        pass

print(f'\nMissing values:\n{df.isnull().sum()[df.isnull().sum()>0]}')

## 📊 2. EDA — Phân tích Chuỗi Thời gian

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(15, 11))

# PM2.5 over time
axes[0].plot(df[TARGET_COL].values, color='steelblue', lw=0.8, alpha=0.9)
axes[0].fill_between(range(len(df)), df[TARGET_COL].values, alpha=0.2, color='steelblue')
axes[0].axhline(df[TARGET_COL].mean(), color='red', linestyle='--', lw=1.5, label=f'Mean={df[TARGET_COL].mean():.1f}')
axes[0].set_title(f'{TARGET_COL.upper()} — Toàn bộ Chuỗi Thời gian')
axes[0].set_ylabel('PM2.5 (µg/m³)')
axes[0].legend()

# Distribution
axes[1].hist(df[TARGET_COL].values, bins=60, color='steelblue',
             edgecolor='white', alpha=0.85)
axes[1].axvline(df[TARGET_COL].median(), color='red', linestyle='--',
                label=f'Median={df[TARGET_COL].median():.1f}')
axes[1].set_title(f'Phân phối {TARGET_COL.upper()}')
axes[1].set_xlabel('µg/m³')
axes[1].legend()

# Monthly/hourly pattern (if datetime available)
try:
    if hasattr(df.index, 'hour'):
        hourly_avg = df.groupby(df.index.hour)[TARGET_COL].mean()
        axes[2].plot(hourly_avg.index, hourly_avg.values, 'o-', color='darkorange', lw=2)
        axes[2].fill_between(hourly_avg.index, hourly_avg.values, alpha=0.2, color='darkorange')
        axes[2].set_title('Trung bình PM2.5 theo Giờ trong ngày')
        axes[2].set_xlabel('Hour of Day')
        axes[2].set_xticks(range(0, 24, 2))
    else:
        raise ValueError()
except:
    rolling = pd.Series(df[TARGET_COL].values).rolling(window=24*7).mean()
    axes[2].plot(df[TARGET_COL].values, alpha=0.3, color='steelblue', lw=0.5, label='Raw')
    axes[2].plot(rolling, color='darkorange', lw=2, label='7-day Rolling Mean')
    axes[2].set_title('Raw vs Rolling Mean')
    axes[2].legend()

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/01_time_series_eda.png', bbox_inches='tight')
plt.show()

In [ ]:
# Select numeric features for multivariate
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove non-feature columns
drop_cols = ['year','month','day','hour','No']
feature_cols = [c for c in numeric_cols if c.lower() not in [d.lower() for d in drop_cols]]
df_feat = df[feature_cols].copy().fillna(method='ffill').fillna(0)

print(f'Feature columns: {feature_cols}')

# Correlation heatmap
fig, ax = plt.subplots(figsize=(10, 8))
corr = df_feat.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlation Heatmap — Beijing Air Quality Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/02_correlation_heatmap.png', bbox_inches='tight')
plt.show()

## ⚙️ 3. Preprocessing — Sliding Window

In [ ]:
# Scale all features
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(df_feat.values)

target_idx = feature_cols.index(TARGET_COL) if TARGET_COL in feature_cols else 0

def create_multivariate_dataset(data, look_back, target_idx):
    X, y = [], []
    for i in range(len(data) - look_back):
        X.append(data[i:i+look_back, :])        # All features as input
        y.append(data[i+look_back, target_idx]) # Only target as output
    return np.array(X), np.array(y)

X_all, y_all = create_multivariate_dataset(data_scaled, LOOK_BACK, target_idx)

split = int(len(X_all) * 0.8)
X_train, X_test = X_all[:split], X_all[split:]
y_train, y_test = y_all[:split], y_all[split:]

print(f'X shape: {X_all.shape}  → (samples, look_back={LOOK_BACK}, features={X_all.shape[2]})')
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

## 🏗️ 4. Multivariate LSTM Model

In [ ]:
n_features = X_train.shape[2]

model = models.Sequential([
    layers.LSTM(128, return_sequences=True, input_shape=(LOOK_BACK, n_features), name='lstm_1'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.LSTM(64, return_sequences=False, name='lstm_2'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, name='forecast')
], name='MultivariateLSTM_AirQuality')

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='mse',
              metrics=['mae'])
model.summary()

## 🚀 5. Training

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=0.1,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor='val_loss'),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=3, min_lr=1e-7, verbose=1)
    ],
    verbose=1
)

## 📈 6. Forecast Visualization & Evaluation

In [ ]:
# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
eps = range(1, len(history.history['loss'])+1)

for ax, (tr, vl, metric) in zip(axes, [
    ('loss','val_loss','MSE Loss'),
    ('mae','val_mae','MAE')
]):
    ax.plot(eps, history.history[tr], 'o-', color='#2196F3', lw=2, label='Train')
    ax.plot(eps, history.history[vl], 's-', color='#FF5722', lw=2, label='Val')
    ax.set_title(f'{metric} — Multivariate LSTM')
    ax.set_xlabel('Epoch'); ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/03_training_history.png', bbox_inches='tight')
plt.show()

In [ ]:
# Inverse-transform predictions back to original scale
y_pred_scaled = model.predict(X_test, verbose=0).flatten()

# Build dummy matrix to inverse transform only target column
dummy = np.zeros((len(y_pred_scaled), n_features))
dummy[:, target_idx] = y_pred_scaled
y_pred_orig = scaler.inverse_transform(dummy)[:, target_idx]

dummy2 = np.zeros((len(y_test), n_features))
dummy2[:, target_idx] = y_test
y_test_orig = scaler.inverse_transform(dummy2)[:, target_idx]

rmse = np.sqrt(mean_squared_error(y_test_orig, y_pred_orig))
mae  = mean_absolute_error(y_test_orig, y_pred_orig)
r2   = r2_score(y_test_orig, y_pred_orig)

print(f'RMSE : {rmse:.4f} µg/m³')
print(f'MAE  : {mae:.4f} µg/m³')
print(f'R²   : {r2:.4f}')

In [ ]:
# Forecast vs Actual plot
show_n = min(500, len(y_test_orig))

fig, axes = plt.subplots(2, 1, figsize=(15, 9))

# Time series
axes[0].plot(y_test_orig[:show_n], color='steelblue', lw=1.5, label='Actual')
axes[0].plot(y_pred_orig[:show_n], color='#FF5722', lw=1.5, linestyle='--', label='Predicted')
axes[0].fill_between(range(show_n), y_test_orig[:show_n], y_pred_orig[:show_n],
                     alpha=0.15, color='gray', label='Error')
axes[0].set_title(f'Dự báo PM2.5 — Actual vs Predicted (RMSE={rmse:.2f}, R²={r2:.4f})')
axes[0].set_xlabel('Time Step'); axes[0].set_ylabel('PM2.5 (µg/m³)')
axes[0].legend()

# Error distribution
errors = y_test_orig - y_pred_orig
axes[1].hist(errors, bins=60, color='darkorange', edgecolor='white', alpha=0.85)
axes[1].axvline(0, color='black', linestyle='--', lw=2)
axes[1].axvline(errors.mean(), color='red', linestyle=':', lw=2, label=f'Mean error={errors.mean():.2f}')
axes[1].set_title('Phân phối Sai số Dự báo (Error Distribution)')
axes[1].set_xlabel('Prediction Error (µg/m³)')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/04_forecast_vs_actual.png', bbox_inches='tight')
plt.show()

In [ ]:
# Scatter: Actual vs Predicted
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test_orig, y_pred_orig, alpha=0.3, s=10, color='steelblue')
mn, mx = min(y_test_orig.min(), y_pred_orig.min()), max(y_test_orig.max(), y_pred_orig.max())
ax.plot([mn, mx], [mn, mx], 'r--', lw=2, label='Perfect Forecast')
ax.set_xlabel('Actual PM2.5 (µg/m³)')
ax.set_ylabel('Predicted PM2.5 (µg/m³)')
ax.set_title(f'Scatter: Actual vs Predicted\nR² = {r2:.4f}')
ax.legend()
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/05_scatter_actual_vs_pred.png', bbox_inches='tight')
plt.show()

In [ ]:
with open(f'{SAVE_DIR}/report.txt', 'w') as f:
    f.write('Beijing Air Quality — Multivariate LSTM\n' + '='*50 + '\n')
    f.write(f'LOOK_BACK     : {LOOK_BACK} hours\n')
    f.write(f'Features      : {feature_cols}\n')
    f.write(f'Total Params  : {model.count_params():,}\n\n')
    f.write(f'Test RMSE     : {rmse:.4f} µg/m³\n')
    f.write(f'Test MAE      : {mae:.4f} µg/m³\n')
    f.write(f'Test R²       : {r2:.4f}\n')

print('✅ Beijing Air Quality Multivariate LSTM Done! Saved to', SAVE_DIR)